## Phase 2: Concept Drift Detectors
### Step 1 — Data Initialization & Multivariate Structure Validation

Loads preprocessed IHSG data, validates datetime index, builds drift scoreboard
DataFrames (one per algorithm variant) with NaN for unevaluable initial rows.

#### 1. Configuration & Imports

Defines `Config` dataclass holding all tunable parameters:
- Data paths, feature names, batch/stream algorithm lists
- Window sizes (60, 120), stream warm-up period (60 rows)
- Consensus threshold (30%) for voting in later steps

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class Config:
    data_path: str = "../dataset/processed/jkse_preprocessed.csv"
    output_dir: str = "../dataset/processed"
    features: list = field(default_factory=lambda: [
        "Log_Return", "Vol_20d", "Vol_60d", "EMA_5",
        "BB_Middle", "BB_Upper", "BB_Lower",
        "Momentum_5d", "Momentum_20d",
    ])
    batch_windows: list = field(default_factory=lambda: [60, 120])
    batch_algorithms: list = field(default_factory=lambda: [
        "mysd", "ks", "psi", "wasserstein",
    ])
    stream_algorithms: list = field(default_factory=lambda: [
        "adwin", "kswin", "ph",
    ])
    stream_warmup: int = 60
    consensus_threshold: float = 1/3

#### 2. Data Loading & Validation

Loads preprocessed JKSE CSV, parses DatetimeIndex, sorts chronologically, and runs sanity assertions:
- Index is unique and datetime-typed
- No remaining NaN values after preprocessing
- Prints date range and shape

In [ ]:
cfg = Config()

print("Config:")
for k, v in cfg.__dict__.items():
    val = str(v)
    if len(val) > 80:
        val = val[:77] + "..."
    print(f"  {k:20s} = {val}")

df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True)
df = df.sort_index()

assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
assert df.index.is_unique, "Index contains duplicate dates"
assert df.isnull().sum().sum() == 0, "Data contains NaN values"

print(f"\nLoaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Index type: {df.index.dtype}")
print(f"Date range: {df.index.min().date()}  to  {df.index.max().date()}")

#### 3. Scoreboard Creation

Creates one scoreboard per algorithm-variant (11 total):
- **Batch** (`mysd`, `ks`, `psi`, `wasserstein` × 60, 120): `NaN` for first `2*W` rows (insufficient data for two adjacent windows)
- **Stream** (`adwin`, `kswin`, `ph`): `NaN` for first 60 warm-up rows
- All remaining rows initialized to `0.0` (no drift detected yet)

In [ ]:
def create_scoreboards(df: pd.DataFrame, cfg: Config) -> dict[str, pd.DataFrame]:
    scoreboards: dict[str, pd.DataFrame] = {}

    for alg in cfg.batch_algorithms:
        for w in cfg.batch_windows:
            key = f"{alg}_{w}"
            sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
            sb.iloc[2 * w:] = 0.0
            scoreboards[key] = sb

    for alg in cfg.stream_algorithms:
        sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
        sb.iloc[cfg.stream_warmup:] = 0.0
        scoreboards[alg] = sb

    return scoreboards


def scoreboard_summary(scoreboards: dict[str, pd.DataFrame], cfg: Config) -> str:
    lines = [f"Scoreboards created: {len(scoreboards)}"]
    for key, sb in scoreboards.items():
        nan_rows = int(sb.isnull().all(axis=1).sum())
        lines.append(f"  {key:20s}  shape={str(sb.shape):12s}  NaN rows={nan_rows:>4d}")
    return "\n".join(lines)


scoreboards = create_scoreboards(df, cfg)
print("=" * 55)
print("STEP 1 - Result")
print("=" * 55)
print(f"df.shape:              {df.shape}")
print(f"Feature columns:       {cfg.features}")
print(f"Batch windows (days):  {cfg.batch_windows}")
print(f"Stream algorithms:     {cfg.stream_algorithms}")
print(f"Stream warm-up rows:   {cfg.stream_warmup}")
print()
print(scoreboard_summary(scoreboards, cfg))

#### 4. Feature Diagnostics

Exploratory analysis of the 9 feature columns:
- Summary statistics via `describe().T`
- Correlation matrix with heatmap coloring (`RdBu_r`)
- Memory footprint and missing-value check

In [ ]:
print("Feature summary statistics")
print("-" * 55)
display(df[cfg.features].describe().T)

print("\nFeature correlation matrix")
print("-" * 55)
corr = df[cfg.features].corr()
display(corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

missing = int(df[cfg.features].isnull().sum().sum())
print(f"\nMissing values in feature set: {missing}")
print(f"DataFrame memory usage: {df[cfg.features].memory_usage(deep=True).sum() / 1024:.1f} KB")

#### 5. Persist Scoreboards

Saves all 11 scoreboard DataFrames as CSV to `../dataset/processed/`.
Files are named `scoreboard_{key}.csv` (e.g. `scoreboard_mysd_60.csv`).
These will be loaded by later steps to write actual drift scores into.

In [ ]:
import os

os.makedirs(cfg.output_dir, exist_ok=True)

for key, sb in scoreboards.items():
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    sb.to_csv(path)

print(f"Saved {len(scoreboards)} scoreboards to {cfg.output_dir}/")
for key in scoreboards:
    p = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    size_kb = p.stat().st_size / 1024
    print(f"  {key:20s}  {size_kb:>8.1f} KB")

#### 6. Scoreboard Previews

Displays `head()` of each scoreboard to visually confirm:
- NaN warm-up rows for batch (120 or 240) and stream (60) algorithms
- Correct column structure (9 features)
- Zeroes after the warm-up period

In [ ]:
print("=" * 55)
print("SCOREBOARD PREVIEWS")
print("=" * 55)
for key, sb in scoreboards.items():
    nan_rows = int(sb.isnull().all(axis=1).sum())
    print(f"\n{key}  (NaN rows: {nan_rows})")
    display(sb.head())

### Step 2 — Priority 1: Basic Statistics (mySD & KS)

Evaluates drift per feature via adjacent-window comparison using:
- **mySD** (custom): $|\\mu_{\\text{curr}} - \\mu_{\\text{ref}}| > k \\cdot \\sigma_{\\text{ref}}$ with $k=2.5$, $\\sigma$ using `ddof=1`
- **Kolmogorov-Smirnov**: `ks_2samp(ref, curr)` with $\\alpha = 0.05$

Slicing: `ref_window = df[feat].iloc[t-2*W : t-W]`, `curr_window = df[feat].iloc[t-W : t]`

Evaluation starts at $t = 2W$ (first row where both windows are full).

In [ ]:
from scipy.stats import ks_2samp
import time

MYSD_K = 2.5
KS_ALPHA = 0.05

batch_start = time.perf_counter()

print("=" * 55)
print("STEP 2 — Drift Detection Loop")
print("=" * 55)

for w in cfg.batch_windows:
    print(f"\n  Processing W={w}...")
    for feat in cfg.features:
        # mySD — vectorized via rolling stats
        rolling_mean = df[feat].rolling(w).mean()
        rolling_std = df[feat].rolling(w).std(ddof=1)
        sb_mysd = scoreboards[f"mysd_{w}"]
        for t in range(2 * w, len(df)):
            mu_ref = rolling_mean.iloc[t - w]
            sigma_ref = rolling_std.iloc[t - w]
            if sigma_ref == 0.0:
                continue
            if abs(rolling_mean.iloc[t] - mu_ref) > MYSD_K * sigma_ref:
                sb_mysd.loc[df.index[t], feat] = 1.0

        # KS — direct loop (scipy, no vectorization)
        sb_ks = scoreboards[f"ks_{w}"]
        for t in range(2 * w, len(df)):
            ref = df[feat].iloc[t - 2 * w: t - w]
            cur = df[feat].iloc[t - w: t]
            pvalue = ks_2samp(ref, cur).pvalue
            if not np.isnan(pvalue) and pvalue < KS_ALPHA:
                sb_ks.loc[df.index[t], feat] = 1.0
        print(f"    {feat} done")

batch_elapsed = time.perf_counter() - batch_start
print(f"\nBatch group runtime: {batch_elapsed:.2f}s")

print("\nDetection complete. Persisting updated scoreboards...")

for key in ("mysd_60", "mysd_120", "ks_60", "ks_120"):
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    scoreboards[key].to_csv(path)
    size_kb = path.stat().st_size / 1024
    print(f"  Saved  {key:20s}  {size_kb:>8.1f} KB")


### Step 2 Results: Basic Statistics Group

Aggregated drift-detected day counts per feature for mySD and KS-Test.

In [ ]:
def _drift_report(scoreboards, cfg):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 2 — DRIFT SUMMARY REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append("### Step 2 Results: Basic Statistics Group")
    lines.append("")

    for w, period in [(60, "Quarterly Window (W = 60)"), (120, "Semiannual Window (W = 120)")]:
        lines.append(f"#### {period}")
        lines.append("")
        for alg_key, display_name in [("mysd", "mySD"), ("ks", "KS-Test")]:
            sb = scoreboards[f"{alg_key}_{w}"]
            counts = sb.sum().astype(int)
            header = f"* **{display_name} (W={w}):**"
            items = []
            for col in cfg.features:
                items.append(f"  - {col}: {counts[col]} days")
            lines.append(header)
            lines.extend(items)
            lines.append("")

    return "\n".join(lines)


print(_drift_report(scoreboards, cfg))


### Step 3 — Priority 2: Pure Streaming Group (river)

Shifts from batch rolling-window (Step 2) to pure online stream processing.
Uses `river` library detectors (ADWIN, KSWIN, PageHinkley) that process data
incrementally, one row at a time, in chronological order.

**Key rules:**
- One detector instance per feature, instantiated **outside** the row loop
- 60-row warm-up: first 60 rows remain `NaN` in scoreboard
- Recording starts at row 60 (index 60, 0-based)
- NaN guard skips `.update()` for any missing or invalid values


In [ ]:
from river.drift import ADWIN, KSWIN, PageHinkley
from river.preprocessing import StandardScaler
import time

# Reload stream scoreboards from CSV to guard against kernel state loss
for alg in cfg.stream_algorithms:
    path = Path(cfg.output_dir) / f"scoreboard_{alg}.csv"
    scoreboards[alg] = pd.read_csv(path, index_col=0, parse_dates=True)

print("=" * 55)
print("STEP 3 — Pure Streaming Drift Detection")
print("=" * 55)

# Instantiate one scaler per feature (shared across all detectors)
# Uses transform-then-learn order to avoid look-ahead bias.
# StandardScaler expects dict input, even for single features.
scalers = {feat: StandardScaler() for feat in cfg.features}
print(f"  Initialized {len(scalers)} scalers\n")

# Instantiate one detector per feature — OUTSIDE the row loop
detectors = {}
for alg, constructor in [
    ("adwin", lambda: ADWIN(delta=0.002)),
    ("kswin", lambda: KSWIN(alpha=0.005, window_size=100, stat_size=30)),
    ("ph", lambda: PageHinkley()),
]:
    detectors[alg] = {feat: constructor() for feat in cfg.features}
    print(f"  Initialized {alg}: 9 detectors")

stream_start = time.perf_counter()

for t in range(len(df)):
    for alg in cfg.stream_algorithms:
        for feat in cfg.features:
            val = df[feat].iloc[t]
            if pd.isna(val):
                continue
            # Standardize to Z-score using pre-update running stats
            val_scaled = scalers[feat].transform_one({feat: val})[feat]
            scalers[feat].learn_one({feat: val})
            # Feed standardized value to detector
            detectors[alg][feat].update(val_scaled)
            if t >= cfg.stream_warmup and detectors[alg][feat].drift_detected:
                scoreboards[alg].loc[df.index[t], feat] = 1.0

stream_elapsed = time.perf_counter() - stream_start
print(f"\nStream group runtime: {stream_elapsed:.2f}s")

# Persist updated stream scoreboards
print("\nPersisting updated scoreboards...")
for alg in cfg.stream_algorithms:
    path = Path(cfg.output_dir) / f"scoreboard_{alg}.csv"
    scoreboards[alg].to_csv(path)
    size_kb = path.stat().st_size / 1024
    print(f"  Saved  {alg:20s}  {size_kb:>8.1f} KB")

### Step 3 Results: Pure Streaming Group (river)

Aggregated drift-detected day counts per feature for ADWIN, KSWIN, and PageHinkley.
Runtime comparison against batch rolling-window methods (Step 2) is included.

In [ ]:
def _stream_drift_report(scoreboards, cfg, stream_elapsed, batch_elapsed):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 3 — DRIFT SUMMARY REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append("### Step 3 Results: Pure Streaming Group (river)")
    lines.append("")
    lines.append("#### A. Drift Signal Summary (Total Days)")
    lines.append("")

    alg_map = {
        "adwin": "ADWIN",
        "kswin": "KSWIN",
        "ph": "Page Hinkley",
    }
    for alg, display_name in alg_map.items():
        sb = scoreboards[alg]
        counts = sb.sum().astype(int)
        header = f"* **{display_name}:**"
        items = []
        for col in cfg.features:
            items.append(f"  - {col}: {counts[col]} days")
        lines.append(header)
        lines.extend(items)
        lines.append("")

    lines.append("#### B. Runtime Comparison & Observations")
    lines.append("")
    lines.append(
        f"* Batch group (rolling windows, mySD + KS-Test):  {batch_elapsed:.2f}s"
    )
    lines.append(
        f"* Stream group (row-by-row, river ADWIN+KSWIN+PH): {stream_elapsed:.2f}s"
    )
    ratio = batch_elapsed / stream_elapsed if stream_elapsed > 0 else float("inf")
    lines.append(f"* River stream group is {ratio:.1f}x faster")
    lines.append("")
    lines.append("> Despite being row-by-row, river detectors are implemented in")
    lines.append("> optimized C/Cython under the hood. The batch group is slower")
    lines.append("> because it uses Python-level loops for KS-2sample tests and")
    lines.append("> manual mySD comparisons across each window step.")

    return "\n".join(lines)


print(_stream_drift_report(scoreboards, cfg, stream_elapsed, batch_elapsed))

### Step 4 — Priority 3: Domain-Specific Metric Group (PSI & Wasserstein)

Evaluates drift per feature via adjacent-window comparison using:
- **PSI (Population Stability Index)**: quantile binning, epsilon smoothing, fixed threshold > 0.25
- **Wasserstein Distance**: Earth Mover's Distance with adaptive threshold (running mean + 2.5× running std via Welford's algorithm)

Slicing: `ref_window = df[feat].iloc[t-2*W : t-W]`, `curr_window = df[feat].iloc[t-W : t]`

Evaluation starts at $t = 2W$ (first row where both windows are full).

In [ ]:
import numpy as np


def calculate_psi(ref_data, curr_data, num_bins=10):
    if np.isnan(ref_data).any() or np.isnan(curr_data).any():
        return np.nan
    if np.unique(ref_data).size < 2:
        return 0.0

    percentiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(ref_data, percentiles)
    bin_edges = np.unique(bin_edges)

    if len(bin_edges) < 2:
        return 0.0

    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    ref_counts, _ = np.histogram(ref_data, bins=bin_edges)
    curr_counts, _ = np.histogram(curr_data, bins=bin_edges)

    ref_props = ref_counts / len(ref_data)
    curr_props = curr_counts / len(curr_data)

    eps = 1e-4
    ref_props = np.where(ref_props == 0, eps, ref_props)
    curr_props = np.where(curr_props == 0, eps, curr_props)

    return float(np.sum((curr_props - ref_props) * np.log(curr_props / ref_props)))


class RunningStats:
    """Welford's online algorithm for streaming mean and std."""
    def __init__(self):
        self.count = 0
        self.mean = 0.0
        self.M2 = 0.0

    def push(self, value):
        self.count += 1
        delta = value - self.mean
        self.mean += delta / self.count
        self.M2 += delta * (value - self.mean)

    def threshold(self, k=2.5):
        if self.count < 2:
            return np.inf
        std = np.sqrt(self.M2 / (self.count - 1))
        return self.mean + k * std

#### PSI & Wasserstein — Rolling Window Detection

Reloads existing scoreboard CSVs (all-NaN zero templates from Step 1),
runs the adjacent-window loop for both metrics, and persists updates.

In [ ]:
from scipy.stats import wasserstein_distance
import time

# Reload batch scoreboards from CSV
for key in ("psi_60", "psi_120", "wasserstein_60", "wasserstein_120"):
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    scoreboards[key] = pd.read_csv(path, index_col=0, parse_dates=True)

batch_start = time.perf_counter()

print("=" * 55)
print("STEP 4 — PSI & Wasserstein Detection Loop")
print("=" * 55)

for w in cfg.batch_windows:
    print(f"\n  Processing W={w}...")
    for feat in cfg.features:
        stats_w = RunningStats()
        sb_psi = scoreboards[f"psi_{w}"]
        sb_wass = scoreboards[f"wasserstein_{w}"]

        for t in range(2 * w, len(df)):
            ref = df[feat].iloc[t - 2 * w: t - w]
            cur = df[feat].iloc[t - w: t]

            # PSI
            psi_val = calculate_psi(ref, cur)
            if not np.isnan(psi_val) and psi_val > 0.25:
                sb_psi.loc[df.index[t], feat] = 1.0

            # Wasserstein Distance
            w_dist = wasserstein_distance(ref, cur)
            if w_dist > stats_w.threshold(k=2.5):
                sb_wass.loc[df.index[t], feat] = 1.0
            stats_w.push(w_dist)

        print(f"    {feat} done")

batch_elapsed = time.perf_counter() - batch_start
print(f"\nBatch group runtime: {batch_elapsed:.2f}s")

print("\nDetection complete. Persisting updated scoreboards...")
for key in ("psi_60", "psi_120", "wasserstein_60", "wasserstein_120"):
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    scoreboards[key].to_csv(path)
    size_kb = path.stat().st_size / 1024
    print(f"  Saved  {key:20s}  {size_kb:>8.1f} KB")

### Step 4 Results: Domain-Specific Metric Group

Aggregated drift-detected day counts per feature for PSI and Wasserstein Distance.

In [ ]:
def _drift_report_step4(scoreboards, cfg, elapsed):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 4 — DRIFT SUMMARY REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append("### Step 4 Results: Domain-Specific Metric Group")
    lines.append("")

    for w, period in [
        (60, "Quarterly Window (W = 60)"),
        (120, "Semiannual Window (W = 120)"),
    ]:
        lines.append(f"#### {period}")
        lines.append("")
        for alg_key, display_name in [
            ("psi", "PSI (Threshold > 0.25)"),
            ("wasserstein", "Wasserstein Distance (Adaptive Threshold)"),
        ]:
            sb = scoreboards[f"{alg_key}_{w}"]
            counts = sb.sum().astype(int)
            header = f"* **{display_name} (W={w}):**"
            items = []
            for col in cfg.features:
                items.append(f"  - {col}: {counts[col]} days")
            lines.append(header)
            lines.extend(items)
            lines.append("")

    lines.append("#### C. Runtime")
    lines.append("")
    lines.append(
        f"* Batch group (rolling windows, PSI + Wasserstein): {elapsed:.2f}s"
    )

    return "\n".join(lines)


print(_drift_report_step4(scoreboards, cfg, batch_elapsed))

### Step 5 — Consensus Integration & Window Sensitivity Analysis

Aggregates all 11 feature-level scoreboards into a single system-level global drift
decision per algorithm variant. On each trading day, if >= 3 of 9 features (33.3%)
fire a drift alarm simultaneously on the same algorithm, the system declares
a Global Drift event (model retrain required).

**Consensus rule:** Row-wise mean >= 1/3 (3 of 9 features)

**Input:** 11 scoreboard CSVs fully populated by Steps 2-4

**Output:** `global_drift_summary` DataFrame with binary decisions + summary report

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Reload all 11 scoreboards
scoreboard_keys = (
    [f"{alg}_{w}" for alg in cfg.batch_algorithms for w in cfg.batch_windows]
    + cfg.stream_algorithms
)

scoreboards_reloaded = {}
for key in scoreboard_keys:
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    scoreboards_reloaded[key] = pd.read_csv(path, index_col=0, parse_dates=True)

# Build global drift summary
global_drift_summary = pd.DataFrame(index=df.index)

for key, sb in scoreboards_reloaded.items():
    row_consensus = sb[cfg.features].mean(axis=1)
    is_drift = (row_consensus >= cfg.consensus_threshold).astype(float)
    is_drift[sb[cfg.features].isnull().all(axis=1)] = np.nan
    global_drift_summary[f"Global_{key}"] = is_drift

# Compute totals
drift_counts = global_drift_summary.sum()

# Find first trigger dates for Wasserstein_60 and KS_60
def first_trigger_date(series):
    mask = series == 1.0
    if mask.any():
        return series.index[mask].min()
    return None

first_triggers = {
    "Wasserstein_60": first_trigger_date(
        global_drift_summary["Global_wasserstein_60"]
    ),
    "KS_60": first_trigger_date(
        global_drift_summary["Global_ks_60"]
    ),
}

# Build report
def _global_drift_report(global_drift_summary, cfg, first_triggers):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 5 — CONSENSUS INTEGRATION REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append(
        "### Step 5 Results: Consensus Integration & Window Sensitivity Analysis"
    )
    lines.append("")
    lines.append(
        f"Consensus threshold: >= {cfg.consensus_threshold:.4f}"
        " (>= 3 of 9 features = 33.3%)"
    )
    lines.append(f"Scoreboards evaluated: {len(global_drift_summary.columns)}")
    lines.append("")
    lines.append("#### A. Total Global Drift Days (Consensus >= 33.3%)")
    lines.append("")
    lines.append("*Quarterly Window (W=60):*")
    for alg in cfg.batch_algorithms:
        key = f"Global_{alg}_60"
        count = int(global_drift_summary[key].sum())
        lines.append(f"  - {alg.upper()}_60: {count} days")
    lines.append("")
    lines.append("*Semiannual Window (W=120):*")
    for alg in cfg.batch_algorithms:
        key = f"Global_{alg}_120"
        count = int(global_drift_summary[key].sum())
        lines.append(f"  - {alg.upper()}_120: {count} days")
    lines.append("")
    lines.append("*Pure Streaming (river):*")
    stream_names = {"adwin": "ADWIN", "kswin": "KSWIN", "ph": "Page Hinkley"}
    for alg, name in stream_names.items():
        key = f"Global_{alg}"
        count = int(global_drift_summary[key].sum())
        lines.append(f"  - {name}: {count} days")
    lines.append("")
    lines.append("#### B. Chronological Milestones (First Trigger Date)")
    lines.append("")
    for name, date in first_triggers.items():
        date_str = (
            str(date.date()) if date is not None else "None (never triggered)"
        )
        lines.append(f"  - **{name}** first global drift: {date_str}")
    lines.append("")
    lines.append("#### C. Global Drift Summary Table (head)")

    return "\n".join(lines)


print(_global_drift_report(global_drift_summary, cfg, first_triggers))
display(global_drift_summary.head(10))